# 01 — Data Loading & Exploratory Data Analysis

This notebook loads the synthetic financial transaction dataset, validates its schema,
and performs exploratory analysis with visualizations.

**Runs on**: Azure Databricks (PySpark) or locally (Pandas).

In [0]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.express as px
from src.data_loader import load_data, preprocess, get_user_data, get_summary_stats

## 1. Load Dataset

In [0]:
DATA_PATH = '../data/virtual_financial_advisor_data.csv'
# On Databricks, use: DATA_PATH = '/dbfs/FileStore/virtual_financial_advisor_data.csv'

df = load_data(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [0]:
df.dtypes

In [0]:
df.describe()

## 2. Preprocess & Explore

In [0]:
df = preprocess(df)
print('Unique users:', df['user_id'].nunique())
print('Date range:', df['date'].min(), 'to', df['date'].max())
print('Categories:', df['category'].unique().tolist())

## 3. Summary Statistics per User

In [0]:
user_stats = []
for uid in sorted(df['user_id'].unique()):
    udf = get_user_data(df, uid)
    stats = get_summary_stats(udf)
    stats['user_id'] = uid
    user_stats.append(stats)

stats_df = pd.DataFrame(user_stats)
stats_df.set_index('user_id', inplace=True)
stats_df

## 4. Visualizations

In [0]:
# Monthly income vs expenses (all users)
df['month_str'] = df['date'].dt.to_period('M').astype(str)
income = df[df['amount'] > 0].groupby('month_str')['amount'].sum().rename('Income')
expenses = df[df['amount'] < 0].groupby('month_str')['amount'].sum().abs().rename('Expenses')
monthly = pd.DataFrame({'Income': income, 'Expenses': expenses}).fillna(0).reset_index()

fig = px.line(monthly, x='month_str', y=['Income', 'Expenses'],
              title='Monthly Income vs Expenses (All Users)',
              labels={'value': 'Amount ($)', 'month_str': 'Month'})
fig.show()

In [0]:
# Category distribution
cat_totals = df.groupby('category')['abs_amount'].sum().reset_index().sort_values('abs_amount', ascending=False)
fig2 = px.bar(cat_totals, x='category', y='abs_amount',
              title='Total Amount by Category',
              labels={'abs_amount': 'Total ($)', 'category': 'Category'})
fig2.show()

In [0]:
# Spending over time - sample user
sample_user = 'user_1'
user_df = get_user_data(df, sample_user)
user_monthly = user_df[user_df['amount'] < 0].groupby('month_str')['amount'].sum().abs().reset_index()

fig3 = px.line(user_monthly, x='month_str', y='amount',
               title=f'Monthly Spending — {sample_user}',
               labels={'amount': 'Spending ($)', 'month_str': 'Month'})
fig3.show()

## 5. PySpark Loading (Databricks Only)

Uncomment the cell below when running on Databricks.

In [0]:
# from src.data_loader import load_data_spark
# sdf = load_data_spark(spark, '/dbfs/FileStore/virtual_financial_advisor_data.csv')
# sdf.printSchema()
# sdf.show(5)
# print(f'Row count: {sdf.count()}')